In [1]:
from _client import Client
import httpx
import json

In [2]:
base_url="https://freva.dkrz.de/api/chatbot/"

In [3]:
url = base_url + "ping"
response=httpx.get(url=url)
print(response.json())

{'endpoints': [{'methods': ['get'], 'name': 'ping', 'params': {}, 'return_type': 'json{version:string,streamvariants:list{string},endpoints:list{name:string,methods:string,params:list{json},returntype:json}}'}, {'methods': ['get'], 'name': 'docs', 'params': {}, 'return_type': 'string'}, {'methods': ['get'], 'name': 'getthread', 'params': {'auth_key': 'string', 'thread_id': 'string'}, 'return_type': 'json{list{variant:streamvariant=string,content:string}}'}, {'methods': ['get'], 'name': 'streamresponse', 'params': {'auth_key': 'string', 'input': 'string', 'thread_id': 'optional{string}'}, 'return_type': 'stream{json{variant:streamvariant=string,content:string}}'}, {'methods': ['get', 'post'], 'name': 'stop', 'params': {'auth_key': 'string', 'thread_id': 'string'}, 'return_type': ''}], 'streamvariants': ['Prompt', 'User', 'Assistant', 'Code', 'CodeOutput', 'Image', 'ServerError', 'OpenAIError', 'CodeError', 'StreamEnd', 'ServerHint'], 'version': '1.8.8'}


In [4]:
auth_key="***REMOVED***"
input_string = "This is a test regarding your capabilities of using the code_interpreter tool and whether it supports matplotlib. Please use the code_interpreter tool to run the following code: \"import numpy as np\nimport matplotlib.pyplot as plt\nt = np.linspace(-2 * np.pi, 2 * np.pi, 100)\nsine_wave = np.sin(t)\nplt.figure(figsize=(10, 5))\nplt.plot(t, sine_wave, label='Sine Wave')\nplt.title('Sine Wave from -2π to 2π')\nplt.xlabel('Angle (radians)')\nplt.ylabel('Sine value')\nplt.axhline(0, color='black', linewidth=0.5, linestyle='--')\nplt.axvline(0, color='black', linewidth=0.5, linestyle='--')\nplt.grid()\nplt.legend()\nplt.show()\"."

In [5]:
url = base_url + "streamresponse"

raw_response = []
with httpx.stream(method="GET", url=url, timeout=None, params={"auth_key":auth_key, "input":input_string}) as r:
    for chunk in r.iter_bytes():
        raw_response.append(chunk)

In [12]:
def process_chunks(chunk:str, partial_response:str=""):
    chunk_split = chunk.split("}{")
    if len(chunk_split) == 1:
        if chunk[0] == "{" and chunk[-1] == "}":
            return [chunk], ""
        elif chunk[0] == "{" and chunk[-1] != "}":
            partial_response = chunk
            return [], partial_response
        elif chunk[-1] == "}":
            partial_response += chunk
            return [partial_response], ""
        else:
            partial_response += chunk
            return [], partial_response
    else:
        complete_parts = []
        for i, part in enumerate(chunk_split):
            if i==0:
                fixed_part = part + "}"
                if part[0] != "{":
                    partial_response += fixed_part
                    complete_parts.append(partial_response)
                    continue
            elif i==len(chunk_split)-1:
                fixed_part = "{" + part
                if part[-1] != "}": 
                    partial_response = fixed_part
                    return complete_parts, partial_response
                return complete_parts, ""
            else:
                fixed_part = "{" + part + "}"
            complete_parts.append(fixed_part)    

In [13]:
complete_response = []
partial_response=""
for i, chunk in enumerate(raw_response):
    chunk_decoded=chunk.decode("utf-8")
    complete_parts, partial_response = process_chunks(chunk_decoded, partial_response)
    complete_response += complete_parts

In [14]:
for i, r in enumerate(complete_response):
    print(i, json.loads(r))

0 {'variant': 'ServerHint', 'content': '{"thread_id": "eMuyDSR2BWXlR4BF9jPgBO5eyjJNEMTY"}'}
1 {'variant': 'Code', 'content': ['', 'call_gCEA1wSBHrDgcb9JINhrZHsv']}
2 {'variant': 'Code', 'content': ['{"', 'call_gCEA1wSBHrDgcb9JINhrZHsv']}
3 {'variant': 'Code', 'content': ['code', 'call_gCEA1wSBHrDgcb9JINhrZHsv']}
4 {'variant': 'Code', 'content': ['import', 'call_gCEA1wSBHrDgcb9JINhrZHsv']}
5 {'variant': 'Code', 'content': [' as', 'call_gCEA1wSBHrDgcb9JINhrZHsv']}
6 {'variant': 'Code', 'content': ['\\n', 'call_gCEA1wSBHrDgcb9JINhrZHsv']}
7 {'variant': 'Code', 'content': [' matplotlib', 'call_gCEA1wSBHrDgcb9JINhrZHsv']}
8 {'variant': 'Code', 'content': [' as', 'call_gCEA1wSBHrDgcb9JINhrZHsv']}
9 {'variant': 'Code', 'content': ['\\', 'call_gCEA1wSBHrDgcb9JINhrZHsv']}
10 {'variant': 'Code', 'content': [' =', 'call_gCEA1wSBHrDgcb9JINhrZHsv']}
11 {'variant': 'Code', 'content': ['.linspace', 'call_gCEA1wSBHrDgcb9JINhrZHsv']}
12 {'variant': 'Code', 'content': ['2', 'call_gCEA1wSBHrDgcb9JINhrZHs

In [21]:
url = base_url + "streamresponse"
r=httpx.request(method="GET", url=url, timeout=None, params={"auth_key":auth_key, "input":input_string})
raw_response = r.text

In [23]:
process_chunks(chunk=raw_response)

(['{"variant":"ServerHint","content":"{\\"thread_id\\": \\"NjZrCmI4k7ZD1CubvzQd4aXf0vIZ48xr\\"}"}',
  '{"variant":"Code","content":["","call_iDB2an4chPqL1RBjNzRkFuvf"]}',
  '{"variant":"Code","content":["{\\"","call_iDB2an4chPqL1RBjNzRkFuvf"]}',
  '{"variant":"Code","content":["code","call_iDB2an4chPqL1RBjNzRkFuvf"]}',
  '{"variant":"Code","content":["\\":\\"","call_iDB2an4chPqL1RBjNzRkFuvf"]}',
  '{"variant":"Code","content":["import","call_iDB2an4chPqL1RBjNzRkFuvf"]}',
  '{"variant":"Code","content":[" numpy","call_iDB2an4chPqL1RBjNzRkFuvf"]}',
  '{"variant":"Code","content":[" as","call_iDB2an4chPqL1RBjNzRkFuvf"]}',
  '{"variant":"Code","content":[" np","call_iDB2an4chPqL1RBjNzRkFuvf"]}',
  '{"variant":"Code","content":["\\\\n","call_iDB2an4chPqL1RBjNzRkFuvf"]}',
  '{"variant":"Code","content":["import","call_iDB2an4chPqL1RBjNzRkFuvf"]}',
  '{"variant":"Code","content":[" matplotlib","call_iDB2an4chPqL1RBjNzRkFuvf"]}',
  '{"variant":"Code","content":[".pyplot","call_iDB2an4chPqL1RBj

In [25]:
url = base_url + "getthread"
r= httpx.get(url=url, params={"thread_id":"eMuyDSR2BWXlR4BF9jPgBO5eyjJNEMTY", "auth_key":auth_key})

In [28]:
process_chunks(r.text)

([],
 '[{"variant":"ServerHint","content":"{\\"thread_id\\": \\"eMuyDSR2BWXlR4BF9jPgBO5eyjJNEMTY\\"}"},{"variant":"User","content":"This is a test regarding your capabilities of using the code_interpreter tool and whether it supports matplotlib. Please use the code_interpreter tool to run the following code: \\"import numpy as np\\nimport matplotlib.pyplot as plt\\nt = np.linspace(-2 * np.pi, 2 * np.pi, 100)\\nsine_wave = np.sin(t)\\nplt.figure(figsize=(10, 5))\\nplt.plot(t, sine_wave, label=\'Sine Wave\')\\nplt.title(\'Sine Wave from -2π to 2π\')\\nplt.xlabel(\'Angle (radians)\')\\nplt.ylabel(\'Sine value\')\\nplt.axhline(0, color=\'black\', linewidth=0.5, linestyle=\'--\')\\nplt.axvline(0, color=\'black\', linewidth=0.5, linestyle=\'--\')\\nplt.grid()\\nplt.legend()\\nplt.show()\\"."},{"variant":"Code","content":["{\\"code\\":\\"import numpy as np\\\\nimport matplotlib.pyplot as plt\\\\nt = np.linspace(-2 * np.pi, 2 * np.pi, 100)\\\\nsine_wave = np.sin(t)\\\\nplt.figure(figsize=(10, 